# Customer Churn Analysis

A small exploratory analysis on the `exported_churn_data.csv` dataset. The data describes subscription customers, their plans, charges, satisfaction and churn behaviour. I've grouped the work into four blocks: customers, products (subscriptions), engagement, and a combined view.

A note on terminology: the dataset is a churn table, so some marketing-style fields (views, clicks, campaigns) aren't present. Where the brief asks for them I use the closest available column as a stand-in and say so inline.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titleweight': 'bold',
})

In [ ]:
# load + a quick look
raw = pd.read_csv('exported_churn_data.csv')
print('shape:', raw.shape)
raw.head()

In [ ]:
# parse the date columns and derive an age + age group from dob
date_cols = ['dob', 'subscription_start_date', 'renewal_date',
             'cancellation_date', 'complaint_date']
for c in date_cols:
    raw[c] = pd.to_datetime(raw[c], errors='coerce')

now = pd.Timestamp.today()
raw['age'] = ((now - raw['dob']).dt.days / 365.25).round().astype('Int64')

def bucket_age(a):
    if pd.isna(a):
        return 'Unknown'
    if a < 30:
        return 'Young'
    if a < 50:
        return 'Adult'
    return 'Senior'

raw['age_group'] = raw['age'].apply(bucket_age)
print('columns:', raw.columns.tolist())
print()
print(raw['age_group'].value_counts())

In [ ]:
# tiny helper so the charts below stay readable
def bar_plot(series, *, title, xlabel, ylabel, horizontal=False, palette='viridis'):
    fig, ax = plt.subplots()
    colors = sns.color_palette(palette, len(series))
    if horizontal:
        s = series.sort_values()
        s.plot.barh(ax=ax, color=colors, edgecolor='black')
        for i, v in enumerate(s):
            ax.text(v, i, f' {v}', va='center', fontweight='bold')
    else:
        bars = series.plot.bar(ax=ax, color=colors, edgecolor='black')
        for i, v in enumerate(series):
            ax.text(i, v, f'{v}', ha='center', va='bottom', fontweight='bold')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.tight_layout()
    plt.show()

## 2. Customer Analysis (Q1-Q10)

**Q1.** How many unique customers are there?

In [ ]:
n_customers = raw['customerid'].nunique()
print(f'Unique customers: {n_customers}')

**Q2.** Distribution of customers across age groups (Young / Adult / Senior).

In [ ]:
age_counts = raw['age_group'].value_counts()
print(age_counts)
print('\n% share:')
print((age_counts / len(raw) * 100).round(1).astype(str) + '%')
bar_plot(age_counts, title='Customers by Age Group',
         xlabel='Age Group', ylabel='Count', palette='Set2')

**Q3.** Customer count by gender.

In [ ]:
gender_counts = raw['gender'].value_counts()
print(gender_counts)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
gender_counts.plot.bar(ax=axes[0], color=['#4c72b0', '#c44e52', '#55a868'], edgecolor='black')
axes[0].set_title('Count by Gender')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
for i, v in enumerate(gender_counts):
    axes[0].text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

gender_counts.plot.pie(ax=axes[1], autopct='%1.1f%%',
                        colors=['#4c72b0', '#c44e52', '#55a868'], startangle=90)
axes[1].set_ylabel('')
axes[1].set_title('Gender Share')
plt.tight_layout()
plt.show()

**Q4.** Average rating (CSAT) per age group.

In [ ]:
rating_by_age = raw.groupby('age_group')['csat_score'].mean().round(2)
print(rating_by_age)
bar_plot(rating_by_age, title='Avg CSAT by Age Group',
         xlabel='Age Group', ylabel='Avg CSAT', palette='Set2')

**Q5.** Average purchase value (monthly charges) by gender.

In [ ]:
spend_by_gender = raw.groupby('gender')['monthly_charges'].mean().round(2)
print(spend_by_gender)
bar_plot(spend_by_gender, title='Avg Monthly Charges by Gender',
         xlabel='Gender', ylabel='Avg Monthly Charges ($)', palette='Set1')

**Q6.** Which country has the most customers?

In [ ]:
by_country = raw['country'].value_counts()
print(by_country)
print(f'\nTop country: {by_country.index[0]} ({by_country.iloc[0]} customers)')
bar_plot(by_country, title='Customers by Country',
         xlabel='Country', ylabel='Count', palette='crest')

**Q7.** Average rating per country.

In [ ]:
rating_by_country = raw.groupby('country')['csat_score'].mean().round(2).sort_values(ascending=False)
print(rating_by_country)
bar_plot(rating_by_country, title='Avg CSAT by Country',
         xlabel='Country', ylabel='Avg CSAT', horizontal=True, palette='coolwarm')

**Q8.** Average engagement (CLTV) per country. CLTV is used here as the engagement proxy.

In [ ]:
eng_by_country = raw.groupby('country')['cltv'].mean().round(2).sort_values(ascending=False)
print(eng_by_country)
bar_plot(eng_by_country, title='Avg Engagement (CLTV) by Country',
         xlabel='Country', ylabel='Avg CLTV', horizontal=True, palette='mako')

**Q9.** Which age group is most engaged?

In [ ]:
eng_by_age = raw.groupby('age_group')['cltv'].mean().round(2).sort_values(ascending=False)
print(eng_by_age)
print(f'\nMost engaged: {eng_by_age.index[0]} (avg CLTV {eng_by_age.iloc[0]})')
bar_plot(eng_by_age, title='Avg Engagement (CLTV) by Age Group',
         xlabel='Age Group', ylabel='Avg CLTV', palette='Set2')

**Q10.** Do male and female customers differ in average rating? Show with a chart.

In [ ]:
m = raw.loc[raw.gender == 'Male', 'csat_score'].dropna()
f = raw.loc[raw.gender == 'Female', 'csat_score'].dropna()
print(f'Male avg:   {m.mean():.2f}  (n={len(m)})')
print(f'Female avg: {f.mean():.2f}  (n={len(f)})')
print(f'Gap:        {abs(m.mean() - f.mean()):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
gender_rating = raw.groupby('gender')['csat_score'].mean()
gender_rating.plot.bar(ax=axes[0], color=['#4c72b0', '#c44e52', '#55a868'], edgecolor='black')
axes[0].set_title('Avg Rating by Gender')
axes[0].set_ylim(0, 25)
for i, v in enumerate(gender_rating):
    axes[0].text(i, v, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

raw.boxplot(column='csat_score', by='gender', ax=axes[1])
axes[1].set_title('Rating Spread by Gender')
axes[1].set_xlabel('Gender')
plt.suptitle('')
plt.tight_layout()
plt.show()

print('Insight: the two genders show a small gap in average CSAT,',
      f'with males at {m.mean():.2f} vs females at {f.mean():.2f}.')

## 3. Product Analysis (Q11-Q20)

The dataset has no product table, so I treat `subscription_type` as the product and `plan_type` as the product category. `complaint_count` stands in for reviews, `cltv` for engagement.

**Q11.** Top 10 most purchased products.

In [ ]:
top_products = raw['subscription_type'].value_counts().head(10)
print(top_products)
bar_plot(top_products, title='Top 10 Products (by purchases)',
         xlabel='Subscription Type', ylabel='Purchases', palette='flare')

**Q12.** Products with the most reviews (complaints used as proxy).

In [ ]:
most_reviewed = raw.groupby('subscription_type')['complaint_count'].sum().sort_values(ascending=False).head(10)
print(most_reviewed)
bar_plot(most_reviewed, title='Top 10 Reviewed Products',
         xlabel='Subscription Type', ylabel='Total Reviews', palette='Blues_r')

**Q13.** Top 10 products by engagement (avg CLTV).

In [ ]:
top_eng = raw.groupby('subscription_type')['cltv'].mean().sort_values(ascending=False).head(10).round(2)
print(top_eng)
bar_plot(top_eng, title='Top 10 Products by Engagement (CLTV)',
         xlabel='Subscription Type', ylabel='Avg CLTV', palette='Greens_r')

**Q14.** Average rating per product category (plan type).

In [ ]:
cat_rating = raw.groupby('plan_type')['csat_score'].mean().round(2).sort_values(ascending=False)
print(cat_rating)
bar_plot(cat_rating, title='Avg Rating by Category',
         xlabel='Plan Type', ylabel='Avg CSAT', horizontal=True, palette='RdYlGn')

**Q15.** Average engagement per category.

In [ ]:
cat_eng = raw.groupby('plan_type')['cltv'].mean().round(2).sort_values(ascending=False)
print(cat_eng)
bar_plot(cat_eng, title='Avg Engagement by Category',
         xlabel='Plan Type', ylabel='Avg CLTV', horizontal=True, palette='mako')

**Q16.** Compare average "views" (monthly charges) vs "clicks" (complaint count) per category.

In [ ]:
cat_metrics = raw.groupby('plan_type').agg(
    avg_charges=('monthly_charges', 'mean'),
    avg_reviews=('complaint_count', 'mean')
).round(2)
print(cat_metrics)

x = np.arange(len(cat_metrics))
w = 0.35
fig, ax = plt.subplots()
ax.bar(x - w/2, cat_metrics['avg_charges'], w, label='Avg Charges (views)', color='#4c72b0', edgecolor='black')
ax.bar(x + w/2, cat_metrics['avg_reviews'], w, label='Avg Reviews (clicks)', color='#c44e52', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(cat_metrics.index)
ax.set_title('Views vs Clicks by Category')
ax.legend()
plt.tight_layout()
plt.show()

**Q17.** 10 lowest-rated products.

In [ ]:
lowest = raw.groupby('subscription_type')['csat_score'].mean().sort_values().head(10).round(2)
print(lowest)
bar_plot(lowest, title='10 Lowest-Rated Products',
         xlabel='Subscription Type', ylabel='Avg CSAT', palette='Reds_r')

**Q18.** 10 highest-rated products.

In [ ]:
highest = raw.groupby('subscription_type')['csat_score'].mean().sort_values(ascending=False).head(10).round(2)
print(highest)
bar_plot(highest, title='10 Highest-Rated Products',
         xlabel='Subscription Type', ylabel='Avg CSAT', palette='Greens_r')

**Q19.** Which category has high "views" but low purchases? Using charges (views) vs CLTV (purchases).

In [ ]:
cat_view = raw.groupby('plan_type').agg(
    avg_charges=('monthly_charges', 'mean'),
    avg_cltv=('cltv', 'mean'),
    n=('customerid', 'count')
).round(2)
cat_view['ratio'] = (cat_view['avg_charges'] / cat_view['avg_cltv']).round(3)
print(cat_view.sort_values('ratio', ascending=False))

x = np.arange(len(cat_view))
w = 0.35
plt.bar(x - w/2, cat_view['avg_charges'], w, label='Avg Charges (views)', color='#4c72b0', edgecolor='black')
plt.bar(x + w/2, cat_view['avg_cltv'], w, label='Avg CLTV (purchases)', color='#55a868', edgecolor='black')
plt.xticks(x, cat_view.index)
plt.title('Views vs Purchases by Category')
plt.legend()
plt.tight_layout()
plt.show()
print('Highest views-to-purchase ratio:', cat_view['ratio'].idxmax())

**Q20.** Visualise product popularity.

In [ ]:
popularity = raw['subscription_type'].value_counts()
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(popularity.index, popularity.values,
              color=sns.color_palette('viridis', len(popularity)), edgecolor='black')
for b, v in zip(bars, popularity.values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), str(v), ha='center', va='bottom', fontweight='bold')
ax.set_title('Product Popularity')
ax.set_xlabel('Subscription Type')
ax.set_ylabel('Customers')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Engagement Analysis (Q21-Q28)

Here `monthly_charges` is the views proxy, `complaint_count` the clicks proxy, `csat_score` the likes proxy and `cltv` the engagement score.

**Q21.** Total views, clicks and likes.

In [ ]:
tot_views = raw['monthly_charges'].sum()
tot_clicks = raw['complaint_count'].sum()
tot_likes = raw['csat_score'].sum()
print(f'Total views (charges): {tot_views:.2f}')
print(f'Total clicks (complaints): {int(tot_clicks)}')
print(f'Total likes (csat): {tot_likes:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, (lbl, val) in zip(axes, [('Views', tot_views), ('Clicks', tot_clicks), ('Likes', tot_likes)]):
    ax.bar(lbl, val, color=['#4c72b0', '#c44e52', '#55a868'][list(axes).index(ax)], edgecolor='black', width=0.5)
    ax.set_title(lbl)
    ax.text(0, val, f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
plt.suptitle('Total Engagement Metrics', fontweight='bold')
plt.tight_layout()
plt.show()

**Q22.** Overall engagement rate (CLTV vs churn score).

In [ ]:
avg_cltv = raw['cltv'].mean()
avg_churn = raw['churn_score'].mean()
eng_rate = avg_cltv / (avg_cltv + avg_churn) * 100
print(f'Avg CLTV: {avg_cltv:.2f}  |  Avg churn score: {avg_churn:.2f}')
print(f'Overall engagement rate: {eng_rate:.2f}%')

plt.pie([eng_rate, 100 - eng_rate], labels=['Engaged', 'At risk'],
        autopct='%1.1f%%', colors=['#55a868', '#c44e52'], startangle=90, explode=(0.05, 0))
plt.title('Overall Engagement Rate')
plt.tight_layout()
plt.show()

**Q23.** Engagement rate per product.

In [ ]:
prod_eng = raw.groupby('subscription_type').agg(
    avg_cltv=('cltv', 'mean'),
    avg_churn=('churn_score', 'mean')
).round(2)
prod_eng['engagement_rate'] = (prod_eng['avg_cltv'] / (prod_eng['avg_cltv'] + prod_eng['avg_churn']) * 100).round(2)
prod_eng = prod_eng.sort_values('engagement_rate', ascending=False)
print(prod_eng[['avg_cltv', 'avg_churn', 'engagement_rate']])

colors = ['#55a868' if v > 50 else '#c44e52' for v in prod_eng['engagement_rate']]
ax = prod_eng['engagement_rate'].plot.bar(color=colors, edgecolor='black')
ax.axhline(50, color='grey', ls='--', alpha=0.6, label='50% mark')
ax.set_title('Engagement Rate by Product')
ax.set_xlabel('Subscription Type')
ax.set_ylabel('Engagement Rate (%)')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Q24.** Most viewed and most clicked products.

In [ ]:
vc = raw.groupby('subscription_type').agg(
    total_charges=('monthly_charges', 'sum'),
    total_clicks=('complaint_count', 'sum')
).round(2)
top_viewed = vc.sort_values('total_charges', ascending=False).head(5)
top_clicked = vc.sort_values('total_clicks', ascending=False).head(5)
print('Most viewed:')
print(top_viewed[['total_charges']])
print('\nMost clicked:')
print(top_clicked[['total_clicks']])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
top_viewed['total_charges'].plot.bar(ax=axes[0], color='#4c72b0', edgecolor='black')
axes[0].set_title('Top 5 Viewed')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
top_clicked['total_clicks'].plot.bar(ax=axes[1], color='#c44e52', edgecolor='black')
axes[1].set_title('Top 5 Clicked')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Q25.** Least engaged products.

In [ ]:
least_eng = raw.groupby('subscription_type')['cltv'].mean().sort_values().head(10).round(2)
print(least_eng)
bar_plot(least_eng, title='10 Least Engaged Products',
         xlabel='Subscription Type', ylabel='Avg CLTV', palette='Reds_r')

**Q26.** Monthly conversion (new subscriptions) trend.

In [ ]:
raw['month'] = raw['subscription_start_date'].dt.to_period('M')
trend = raw.groupby('month').agg(
    new_subs=('customerid', 'count'),
    avg_charges=('monthly_charges', 'mean')
).round(2)
print(trend.tail(12))

months = [str(m) for m in trend.index]
fig, ax1 = plt.subplots(figsize=(13, 6))
ax1.bar(months, trend['new_subs'], color='#4c72b0', alpha=0.7, label='New subs')
ax1.set_ylabel('New Subscriptions', color='#4c72b0')
ax1.set_xticklabels(months, rotation=45, ha='right', fontsize=8)
ax2 = ax1.twinx()
ax2.plot(months, trend['avg_charges'], color='#c44e52', marker='o', label='Avg charges')
ax2.set_ylabel('Avg Monthly Charges', color='#c44e52')
ax1.set_title('Monthly Conversion Trend')
lines = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labels, loc='upper left')
plt.tight_layout()
plt.show()

**Q27.** Click-through rate per campaign. No Campaign ID exists, so `contract_type` is used as a campaign proxy. CTR = total clicks / customers.

In [ ]:
ctr = raw.groupby('contract_type').agg(
    clicks=('complaint_count', 'sum'),
    customers=('customerid', 'count')
).round(2)
ctr['CTR'] = (ctr['clicks'] / ctr['customers']).round(2)
print(ctr)
bar_plot(ctr['CTR'], title='CTR by Contract Type (campaign proxy)',
         xlabel='Contract Type', ylabel='CTR', palette='Set1')

**Q28.** Top and lowest performing campaigns (by avg CLTV per customer).

In [ ]:
perf = raw.groupby('contract_type').agg(
    total_cltv=('cltv', 'sum'),
    avg_churn=('churn_score', 'mean'),
    customers=('customerid', 'count'),
    revenue=('monthly_charges', 'sum')
).round(2)
perf['score'] = (perf['total_cltv'] / perf['customers']).round(2)
perf = perf.sort_values('score', ascending=False)
print(perf)
print(f"\nBest campaign: {perf.index[0]}\nWorst campaign: {perf.index[-1]}")

cols = ['#55a868' if v == perf['score'].max() else '#c44e52' if v == perf['score'].min() else '#dd8452' for v in perf['score']]
perf['score'].plot.bar(color=cols, edgecolor='black')
plt.title('Campaign Performance Score')
plt.xlabel('Contract Type')
plt.ylabel('Avg CLTV per customer')
plt.tight_layout()
plt.show()

## 5. Combined Analysis (Q29-Q30)

**Q29.** Is there a relationship between engagement (CLTV) and rating (CSAT)? Scatter + correlation.

In [ ]:
valid = raw[['cltv', 'csat_score']].dropna()
corr = valid['cltv'].corr(valid['csat_score'])
print(f'Correlation (CLTV vs CSAT): {corr:.3f}')

plt.figure(figsize=(9, 6))
sns.regplot(data=valid, x='cltv', y='csat_score', scatter_kws={'alpha': 0.7}, line_kws={'color': 'red'})
plt.title('Engagement vs Rating')
plt.xlabel('CLTV (engagement)')
plt.ylabel('CSAT (rating)')
plt.tight_layout()
plt.show()
print('Insight: the correlation is weak/moderate, meaning higher engagement',
      'does not automatically mean a higher rating.')

**Q30.** Correlation matrix (heatmap) across Views, Clicks, Likes, Rating, Duration and Purchase.

In [ ]:
# build a clean numeric set, mapping the brief's marketing fields onto what we have
heat = raw[['monthly_charges', 'complaint_count', 'csat_score', 'cltv', 'churn_score', 'age']].dropna()
heat = heat.rename(columns={
    'monthly_charges': 'Views', 'complaint_count': 'Clicks',
    'csat_score': 'Likes', 'cltv': 'Purchase',
    'churn_score': 'Churn', 'age': 'Age'
})

plt.figure(figsize=(9, 7))
sns.heatmap(heat.corr(), annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

print('Business notes:')
print('- CLTV (purchase) tracks loosely with monthly charges (views).')
print('- Churn score is negatively tied to engagement, as expected.')
print('- CSAT (rating) is only weakly linked to the other metrics.')